# Experiments using Le World Model

## Preparation

Convert config and weights to checkpoint

In [ ]:
import json
import os
from pathlib import Path

import stable_worldmodel as swm
import torch
from hydra.utils import instantiate

from jepa import JEPA

os.environ["STABLEWM_HOME"] = Path.cwd().as_posix()
src = Path(swm.data.utils.get_cache_dir(), "pusht")
out = Path(swm.data.utils.get_cache_dir(), "pusht", "lewm_object.ckpt")

cfg = json.loads((src / "config.json").read_text())

model = JEPA(
    encoder=instantiate(cfg["encoder"]),
    predictor=instantiate(cfg["predictor"]),
    action_encoder=instantiate(cfg["action_encoder"]),
    projector=instantiate(cfg["projector"]),
    pred_proj=instantiate(cfg["pred_proj"]),
)

state_dict = torch.load(src / "weights.pt", map_location="cpu", weights_only=False)
translated_state_dict = {}
for key, tensor in state_dict.items():
    new_key = key

    # Target the mismatched ViT layers
    if "encoder.encoder.layer." in key:
        # Fix the base layer prefix
        new_key = new_key.replace("encoder.encoder.layer.", "encoder.layers.")

        # Translate Attention Q, K, V Projections
        new_key = new_key.replace("attention.attention.query", "attention.q_proj")
        new_key = new_key.replace("attention.attention.key", "attention.k_proj")
        new_key = new_key.replace("attention.attention.value", "attention.v_proj")

        # Translate Attention Output
        new_key = new_key.replace("attention.output.dense", "attention.o_proj")

        # Translate MLP / FeedForward layers
        new_key = new_key.replace("intermediate.dense", "mlp.fc1")
        new_key = new_key.replace("output.dense", "mlp.fc2")

    translated_state_dict[new_key] = tensor

model.load_state_dict(translated_state_dict, strict=True)
out.parent.mkdir(parents=True, exist_ok=True)
torch.save(model, out)

01:04:55 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}


## Evaluation

In [ ]:
!python eval.py --config-name=pusht.yaml policy=pusht/lewm